# Feature Engineering Explanation

This notebook explains what features the `feature_engineering.ipynb` notebook created.

We start from the basic merged dataset (`finland_electricity_predict_dataset.csv`)
and show step-by-step how each feature is built.

**Why this matters:** Understanding your features helps you debug, improve predictions, and trust your model.

## 0. Load the raw merged dataset

This is what `03_data_cleaning_and_alignment.ipynb` produced: price + temperature + wind all merged by time.

In [ ]:
import pandas as pd
import numpy as np
import holidays
import matplotlib.pyplot as plt

# Load the raw merged data (output from 03_data_cleaning_and_alignment.ipynb)
df_raw = pd.read_csv('finland_electricity_predict_dataset.csv', parse_dates=['datetime'])

print('Raw dataset shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
print('\nFirst few rows:')
df_raw.head()

## 1. What is a Feature?

A **feature** is a column that the model uses as **input** to learn the pattern.

A **target** is the column the model tries to predict (in our case: `price`).

Raw dataset has only 3 columns:
- `datetime`: when the price happened
- `price`: the target (what we want to predict)
- `Air temperature mean [°C]`: weather feature
- `Wind speed mean [m/s]`: weather feature

**Problem:** Only 3 features is too little. The model cannot find good patterns.

**Solution:** Create more features (特征工程 = feature engineering).

## 2. Type 1: Temporal Features (时间特征)

Idea: Electricity prices follow daily and seasonal patterns.

Example: Peak hours (晚高峰, 早高峰) have higher prices.

In [ ]:
df = df_raw.copy()
df = df.sort_values('datetime').reset_index(drop=True)

# Extract time components from datetime
df['hour'] = df['datetime'].dt.hour                    # 0-23: hour of day
df['day_of_week'] = df['datetime'].dt.dayofweek       # 0=Mon, 6=Sun: day of week
df['day_of_month'] = df['datetime'].dt.day             # 1-31: which day in the month
df['month'] = df['datetime'].dt.month                 # 1-12: which month
df['week_of_year'] = df['datetime'].dt.isocalendar().week.astype(int)  # 1-52: week number
df['quarter'] = df['datetime'].dt.quarter             # 1-4: quarter of year
df['year'] = df['datetime'].dt.year                   # 2023, 2024, etc.

# Season: map month to season (季节)
# Winter=1, Spring=2, Summer=3, Fall=4
season_map = {12: 1, 1: 1, 2: 1, 3: 2, 4: 2, 5: 2,
              6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4}
df['season'] = df['month'].map(season_map)

# Binary flags (标志位): is this a special time?
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)  # Weekend = higher chance of low prices
df['is_peak_hour'] = (df['hour'].isin(range(7, 10)) | df['hour'].isin(range(17, 21))).astype(int)  # Peak hours = 7-10, 17-21
df['is_night_hour'] = (df['hour'].isin(range(23, 24)) | df['hour'].isin(range(0, 6))).astype(int)  # Night = 23-6

print('Temporal features created:')
print(df[['datetime', 'hour', 'day_of_week', 'month', 'season', 'is_weekend', 'is_peak_hour']].head(10))

## 3. Type 2: Cyclic Encoding (周期编码)

**Problem:** Trees don't understand that hour 23 and hour 0 are adjacent.

If you just use 'hour' as 0-23, the model thinks hour 0 is far from hour 23.

**Solution:** Convert to sine/cosine (正弦/余弦) so the model understands circularity.

Example: hour 0 (midnight) should be close to hour 23 (11pm) because they are 1 hour apart.

In [ ]:
def cyclic_encode(series, max_val):
    # Convert a cyclic value (0 to max_val) to sine and cosine.
    # This maps a circle onto a 2D plane.
    angle = 2 * np.pi * series / max_val
    return np.sin(angle), np.cos(angle)

# Apply to hour (0-23 repeats every day)
df['hour_sin'], df['hour_cos'] = cyclic_encode(df['hour'], 24)
# Apply to day of week (0-6 repeats every week)
df['day_of_week_sin'], df['day_of_week_cos'] = cyclic_encode(df['day_of_week'], 7)
# Apply to month (1-12 repeats every year)
df['month_sin'], df['month_cos'] = cyclic_encode(df['month'], 12)
# Apply to week of year (1-52 repeats every year)
df['week_of_year_sin'], df['week_of_year_cos'] = cyclic_encode(df['week_of_year'], 52)

print('Cyclic features created (now the model understands circularity):')
print(df[['hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos']].head(8))

# Visual: plot hour_sin and hour_cos to see they form a circle
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df['hour_sin'], df['hour_cos'], c=df['hour'], cmap='viridis', s=5)
ax.set_xlabel('hour_sin')
ax.set_ylabel('hour_cos')
ax.set_title('Cyclic encoding of hours: forms a circle')
plt.colorbar(ax.collections[0], ax=ax, label='Hour')
plt.show()

## 4. Type 3: Holiday Flag (节假日标志)

**Idea:** Holidays and weekends have lower electricity demand, so prices may differ.

We use the `holidays` library to get all official Finnish holidays.

In [ ]:
# Get all Finnish holidays for the years in our data
years = df['datetime'].dt.year.unique().tolist()
fi_holidays = holidays.Finland(years=years)

print('Finnish holidays in our data:')
for date in sorted(fi_holidays.keys())[:10]:  # Show first 10
    print(f'{date}: {fi_holidays[date]}')

# Create a flag: 1 if this hour is on a holiday, 0 otherwise
df['is_holiday'] = df['datetime'].dt.date.astype(str).isin(
    [str(d) for d in fi_holidays.keys()]
).astype(int)

# Combined flag: any non-working hour (weekend OR holiday)
df['is_non_working'] = ((df['is_weekend'] == 1) | (df['is_holiday'] == 1)).astype(int)

print('\nHoliday features:')
holiday_counts = df.groupby('is_holiday')['datetime'].count()
print('Non-holiday hours:', holiday_counts.get(0, 0))
print('Holiday hours:', holiday_counts.get(1, 0))

print('\nExamples of holiday hours:')
print(df[df['is_holiday'] == 1][['datetime', 'is_holiday', 'is_non_working']].head(8))

## 5. Type 4: Lag Features (滞后特征) — MOST IMPORTANT

**Idea:** Electricity price follows its own history. The price yesterday is a strong predictor of today.

**Lag features** are previous prices used as inputs:
- `price_lag_1h`: price 1 hour ago
- `price_lag_24h`: price 24 hours ago (same hour yesterday)
- `price_lag_168h`: price 168 hours ago (same hour last week) — VERY IMPORTANT for weekly pattern

**Why lag features are so powerful:**

Time series often have strong autocorrelation (自相关性).

If you know the last price, you can often guess the next price with high accuracy.

In [ ]:
# Create lag features: shift the price column backward by N hours
for h in [1, 2, 3, 6, 12, 24, 48, 168]:
    df[f'price_lag_{h}h'] = df['price'].shift(h)  # shift(h) moves the column UP by h rows

print('Lag features created:')
print(df[['datetime', 'price', 'price_lag_1h', 'price_lag_24h', 'price_lag_168h']].head(200).tail(10))

# Notice: first 168 rows have NaN because we don't have 168 hours before them
print('\nMissing values in lag features (first 200 rows):')
print(df[['price_lag_1h', 'price_lag_24h', 'price_lag_168h']].head(200).isna().sum())

## 6. Type 5: Rolling Features (滚动统计特征)

**Idea:** Price trends matter. If prices have been rising, they may continue to rise.

**Rolling features** compute statistics over a moving window:
- `price_rolling_mean_24h`: average price in the last 24 hours
- `price_rolling_std_24h`: volatility (variation) in the last 24 hours
- `price_rolling_min/max_24h`: price range in the last 24 hours
- `price_rolling_mean_168h`: average price in the last 7 days

Higher volatility might signal price spikes ahead.

In [ ]:
# 24-hour rolling statistics on price
df['price_rolling_mean_24h'] = df['price'].shift(1).rolling(24).mean()  # shift(1) to avoid leakage
df['price_rolling_std_24h'] = df['price'].shift(1).rolling(24).std()
df['price_rolling_min_24h'] = df['price'].shift(1).rolling(24).min()
df['price_rolling_max_24h'] = df['price'].shift(1).rolling(24).max()

# 7-day (168h) rolling mean on price
df['price_rolling_mean_168h'] = df['price'].shift(1).rolling(168).mean()

# 24-hour rolling mean on temperature
df['temp_rolling_mean_24h'] = df['temp'].shift(1).rolling(24).mean()

print('Rolling features created:')
print(df[['datetime', 'price', 'price_rolling_mean_24h', 'price_rolling_std_24h', 'price_rolling_mean_168h']].iloc[200:210])

# Visualize: show how rolling mean smooths price volatility
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['datetime'].iloc[1000:1200], df['price'].iloc[1000:1200], label='Actual price', alpha=0.6)
ax.plot(df['datetime'].iloc[1000:1200], df['price_rolling_mean_24h'].iloc[1000:1200], label='24h rolling mean', linewidth=2)
ax.set_xlabel('datetime')
ax.set_ylabel('Price')
ax.set_title('Rolling mean smooths the price signal')
ax.legend()
plt.show()

## 7. Type 6: Weather-Derived Features (天气衍生特征)

**Idea:** Raw temperature and wind speed are useful, but we can create smarter versions.

**Heating Degree Hours (HDD):**
- When it's cold, people need more heating, so more electricity is used.
- HDD = max(0, 17°C - actual_temp)
- If temp = 10°C, HDD = 7; if temp = 20°C, HDD = 0

**Wind Power Proxy:**
- Wind turbine output scales with wind³ (cubic), not linear.
- More wind = much more power (because power ∝ wind³)

**Lagged Weather:**
- Weather effects are not immediate. Yesterday's cold may affect today's price.
- Lag weather features to capture delayed effects.

In [ ]:
# Rename weather columns for clarity
df = df.rename(columns={
    'Air temperature mean [°C]': 'temp',
    'Wind speed mean [m/s]': 'wind_speed',
})

# Heating Degree Hours: how much colder than comfort threshold
# Higher HDD = colder = more heating demand = higher electricity price
df['HDD'] = (17 - df['temp']).clip(lower=0)  # clip ensures no negative values

# Wind power proxy: turbine power scales with wind³, capped at rated speed (13 m/s)
# When wind exceeds 13 m/s, the turbine is at max output
df['wind_power_proxy'] = df['wind_speed'].clip(upper=13) ** 3

# Lagged temperature (yesterday and last week)
df['temp_lag_24h'] = df['temp'].shift(24)   # temperature 24 hours ago
df['temp_lag_168h'] = df['temp'].shift(168)  # temperature 1 week ago

print('Weather-derived features created:')
print(df[['datetime', 'temp', 'wind_speed', 'HDD', 'wind_power_proxy', 'temp_lag_24h']].head(8))

# Visualize HDD: it captures the heating demand signal
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['datetime'].iloc[0:500], df['temp'].iloc[0:500], label='Temperature', alpha=0.7)
ax.plot(df['datetime'].iloc[0:500], df['HDD'].iloc[0:500], label='HDD (heating demand)', alpha=0.7)
ax.set_xlabel('datetime')
ax.set_ylabel('°C or HDD')
ax.set_title('HDD captures when heating is needed (high HDD = very cold)')
ax.legend()
plt.show()

## 8. Summary: All Features at a Glance

In [ ]:
# Show the final feature set
feature_cols = [col for col in df.columns if col not in ['datetime', 'price']]

print('Total number of features:', len(feature_cols))
print('\nAll features:')
for i, col in enumerate(feature_cols, 1):
    print(f'{i:2d}. {col}')

print('\n--- Feature Categories ---')
print('Temporal (time of day):', [c for c in feature_cols if c in ['hour', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter', 'year', 'season']])
print('Cyclic (periodic):', [c for c in feature_cols if 'sin' in c or 'cos' in c])
print('Flags (binary):', [c for c in feature_cols if c.startswith('is_')])
print('Lag (history):', [c for c in feature_cols if 'lag' in c])
print('Rolling (trend):', [c for c in feature_cols if 'rolling' in c])
print('Weather:', [c for c in feature_cols if c in ['temp', 'wind_speed', 'HDD', 'wind_power_proxy', 'temp_rolling_mean_24h', 'temp_lag_24h', 'temp_lag_168h']])

# Save the full engineered dataset
df.to_csv('finland_electricity_features_v2_with_explanation.csv', index=False)
print('\n✓ Saved engineered features to: finland_electricity_features_v2_with_explanation.csv')

## 9. Key Takeaways (关键要点)

1. **Temporal features** help the model learn daily/seasonal patterns (知道几点、几月)
2. **Cyclic encoding** lets the model understand that time is circular (12点和0点很接近)
3. **Lag features** are the most important because price depends on past price (历史价格最重要)
4. **Rolling features** capture trends and volatility (趋势和波动性)
5. **Weather features** are domain knowledge encoded (冷 → 加热 → 用电 → 价格高)
6. **All together:** 42 features instead of 2, and the model can learn much better patterns.

Next step: Feed these 42 features into your XGBoost model to predict the price!